In [ ]:
!pip install transformers

In [ ]:
! pip install "transformers[torch]"

In [3]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [4]:
train_set = pd.read_csv("samsum-train.csv")
val_set = pd.read_csv("samsum-validation.csv")

In [5]:
print(train_set.shape)
print(val_set.shape)

(14732, 3)
(818, 3)


In [6]:
# random sampling of data
train_data = train_set.sample(n = 4000, random_state =42).reset_index(drop = True)
val_data = val_set.sample(n = 500, random_state = 42).reset_index(drop = True)

### Data Pre-Processing

In [7]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # linespaces
    text = re.sub(r"\s+", " ", text)# whitespaces removal
    text = re.sub(r"<.*?>", " ", text) # html tags
    text = text.strip().lower()

    return text

In [8]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["dialogue"] = train_data["summary"].apply(clean_data)

In [9]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

### Tokenize

In [2]:
tokenizer = T5Tokenizer.from_pretrained("t5-small") # import tokenizer from t5-small model.

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

C:\Users\Ayush Singh\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ayush Singh\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [15]:
# convert raw data => tokens for fine tuning our model

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    # padding -- means make the input lenght of each input same and equal to max-length
    # max-lenght -- each input have 512 tokens or input length = 512
    # truncation -- means agr kisi input ki length 512 se jyada hai to usse chhota karke 512 kardo

    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"] # input_ids means token ids # add target tokens into inputs as labels
    # convert input and target into single unit
    return inputs

In [16]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist() # with hugging face transformer list input is more compatible
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [17]:
train_dataset[0]

# attention_mask - means that kaha par true token hai aur kaha par padding hai 
# 1 means true token and 0 means padding 
# input_ids = tokens for dialogue text input
# labels = tokens for target or summary text input

# in input_ids and labels 1 at last means end of sequence
# len(train_dataset[0]["input_ids"]) will be 512

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

### Working with our model

In [18]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
# defining device for training

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device", device)
model.to(device) # means model device naam ke device me train karna hai

device cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6, # we can increase epochs for better performance of model
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch", # evaluate the model after each epoch
    save_strategy = "epoch", # save the model after each epoch

    warmup_steps = 500 # learning rate reaching from 0 to default value in how many steps
)

In [21]:
# this Trainer class provides an API for feature-complete training in pytorch, it supports distributed training
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
# train the model
trainer.train()

In [ ]:
# saving the model

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
# using the saved model

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")

### Testing the summarization logic

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean the dialogue

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensor = "pt" # pt means pytorch tensor 
    )

    # generate the summary => it will generate the summary token ids
    targets = model.generate(
        # targets is the token ids for 
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4, # it means that model will return 4 different seq of outputs and then it will compare those 4 o/p and gives the best as output 
        early_stopping = True
    )

    # output token ids converted into summary text => decoding

    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # special tokens like EOS, Separator
    return summary